# MLP Optimize

## Notes
- This notebook aims to be the more optimized version of `MLP_Manual.ipynb` choosing the best:
    - Epoch
        - `[50, 100, 150, 200]`
    - Batch
        - `[len(data) / 100, 50, 10, 5, 2, 1]`
    - Learning Rate
        - `[1e-2, 1e-3]`
    - Weight Decay
        - `[1e-2, 1e-3]`
- About 96 runs
- The "best" run is the run with the highest $\text{F1}$.

## File Conversion

### Imports

In [1]:
# General Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from math import ceil
from time import perf_counter
from datetime import datetime

import json

In [2]:
# Load the ECG
TRAIN_ECG  = np.load('train_ecg.npz')
TRAIN_ECGx = TRAIN_ECG['X']
TRAIN_ECGy = TRAIN_ECG['y']

VALID_ECG  = np.load('validation_ecg.npz')
VALID_ECGx = VALID_ECG['X']
VALID_ECGy = VALID_ECG['y']

# Convert the dictionary-like npz contents into a Pandas DataFrame
# Assumes arrays have the same lengths/shapes
df_train = pd.DataFrame(TRAIN_ECGx, columns=[f'x_{i+1}' for i in range(TRAIN_ECGx.shape[1])])
df_train['target'] = TRAIN_ECGy

df_valid = pd.DataFrame(VALID_ECGx, columns=[f'x_{i+1}' for i in range(VALID_ECGx.shape[1])])
df_valid['target'] = VALID_ECGy

In [3]:
{
    'df_train': df_train.shape,
    'df_valid': df_valid.shape,
}

{'df_train': (51182, 253), 'df_valid': (15367, 253)}

In [4]:
df_train['target'].value_counts()

target
0    46397
1     4785
Name: count, dtype: int64

In [5]:
df_valid['target'].value_counts()

target
0    13892
1     1475
Name: count, dtype: int64

## MLP NN

In [6]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, f1_score

In [7]:
print('CUDA:', torch.version.cuda, '| Available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0))

CUDA: 13.4 | Available: True | Device: NVIDIA GeForce RTX 4070 Laptop GPU


In [8]:
SEED = 825
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED) # Set the random seed for reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Set the device to GPU if available, otherwise CPU

# Class wights for training set frequencies
class_counts = np.bincount(TRAIN_ECGy)
class_weights = len(TRAIN_ECGy) / (len(class_counts) * class_counts)
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
    ).to(device)
print(f"Class weights: \n{class_weights}")

criterion      = nn.CrossEntropyLoss(weight=class_weights) # Loss Function for multi-class classification
eval_criterion = nn.CrossEntropyLoss()

Class weights: 
tensor([0.5516, 5.3482], device='cuda:0')


In [9]:
feature_cols = [c for c in df_train.columns if c != 'target']
scaler = StandardScaler().fit(df_train[feature_cols])

def to_tensor_dataset(df):
    '''
    Substitutes **W11_D2**'s `balanced_subset()` function. \n
    Converts a DataFrame into a PyTorch TensorDataset.
    '''
    # Extract features and labels
    X = torch.tensor(scaler.transform(df[feature_cols]), dtype=torch.float32)
    y = torch.tensor(df['target'].values, dtype=torch.long)
    
    return TensorDataset(X, y)

In [10]:
train_data = to_tensor_dataset(df_train)
valid_data = to_tensor_dataset(df_valid)

train_loader = DataLoader(train_data, batch_size=512, shuffle=False)
valid_loader = DataLoader(valid_data, batch_size=512, shuffle=False)

In [11]:
print(f"Train DataLoader: {len(train_loader)} batches, each with {train_loader.batch_size} samples")
print(f"Validation DataLoader: {len(valid_loader)} batches, each with {valid_loader.batch_size} samples")

Train DataLoader: 100 batches, each with 512 samples
Validation DataLoader: 31 batches, each with 512 samples


### `evaluate(model, loader)` Function

In [12]:
def evaluate(model, loader):
    model.eval()
    total, probs, labels = 0.0, [], []
    with torch.inference_mode():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            total += eval_criterion(logits, y_batch).item() * len(y_batch) # criterion(logits, y_batch).item() * len(y_batch)
            probs.append(torch.softmax(logits, dim=1)[:, 1].cpu())  # P(abnormal)
            labels.append(y_batch.cpu())
    return total / len(loader.dataset), torch.cat(probs).numpy(), torch.cat(labels).numpy()

## Parameters

In [13]:
EPOCHS       = [50, 100, 150, 200]
ALPHAS       = [1e-2, 1e-3]
WEIGHT_DECAY = [1e-2, 1e-3]

def batchSizes(data):
    return [ceil(len(data)/100), ceil(len(data)/50), ceil(len(data)/10), ceil(len(data)/5), ceil(len(data)/2), len(data)]

### `fit(model_class)` Function

In [14]:
def fit(
        model_class, 
        data,
        ITER_epochs,
        ITER_batchSize,
        ITER_alpha,
        ITER_weight_decay):
    torch.manual_seed(SEED)
    
    # Create instance of model
    model = model_class().to(device)
    loader = DataLoader(
        data, 
        batch_size=ITER_batchSize,
        shuffle   =False
    )
    optimizer = torch.optim.Adam(
        model.parameters(), 
        lr          =ITER_alpha,
        weight_decay=ITER_weight_decay)

    epoch_log, loss_log, = [], []
    for epoch in range(ITER_epochs):
        model.train()
        total_loss = 0
        epoch_start = perf_counter() # Start timer
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            # Backpropigation
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        loss_proportion = total_loss / len(loader)
        epoch_elapsed = perf_counter() - epoch_start # End timer
        if (epoch + 1) % 25 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{ITER_epochs}, Loss: {loss_proportion:.4f} | {epoch_elapsed:.3f}s")
        epoch_log.append(epoch+1)
        loss_log.append(loss_proportion)
    return model, epoch_log, loss_log

### MLP Class

In [15]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()       # Flattens the input tensor to a 1D tensor
        self.hidden  = nn.Linear(252, 32) # Mapping 252 input features (x_n) to 32 hidden neurons
        self.output  = nn.Linear(32, 2)   # Mapping 32 hidden neurons to 2 output classes (normal/abnormal)
    
    def forward(self, x):
        z = self.hidden(self.flatten(x))
        h = torch.relu(z)
        return self.output(h)

### Training and Validation

In [16]:
from itertools import product
import gc

In [17]:
BATCHES = batchSizes(train_data)

BATCHES, EPOCHS, ALPHAS, WEIGHT_DECAY
params = (BATCHES, EPOCHS, ALPHAS, WEIGHT_DECAY)

In [18]:
device # Check

device(type='cuda')

In [ ]:
# %%script false --no-raise-error
runLog = []
best_F1 = -1
for i, (batch_size_, epoch_, alpha_, weightDecay_) in enumerate(product(*params)):
    # Time
    model_now   = datetime.now()
    model_start = perf_counter() # Start timer
    print(
    f""" ==========     ==========     ==========     ==========
ITERATION [{i+1}/{len(BATCHES)*len(EPOCHS)*len(ALPHAS)*len(WEIGHT_DECAY)}]: {model_now.strftime("%I:%M:%S %p")}
    Batch Size:   {batch_size_}
    Epochs:       {epoch_}
    α:            {alpha_}
    Weight Decay: {weightDecay_}
    """
    )

    # Training
    MLP_MODEL, MLP_MODEL_epoch_log, MLP_MODEL_loss_log = fit(
        MLP, train_data,
        ITER_epochs      =epoch_,
        ITER_batchSize   =batch_size_,
        ITER_alpha       =alpha_,
        ITER_weight_decay=weightDecay_
    )

    # Training Loss / Epochs
    Path("images/optimize").mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(8, 5))
    plt.plot(MLP_MODEL_epoch_log, MLP_MODEL_loss_log, label='Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f"Training Loss over Epochs\nBatch Size = {batch_size_}, Epochs = {epoch_}, Alpha = {alpha_}, Weight Decay = {weightDecay_}")
    plt.legend()
    plt.grid(True)
    plt.savefig(f'images/optimize/MLP_Optimize-Training_Loss_over_Epochs-_{i}_-batchSize-{batch_size_}_epoch-{epoch_}_alpha-{alpha_}_weightDecay{weightDecay_}.png', dpi=300, bbox_inches='tight')

    # Evaluation
    valid_loss, valid_probs, valid_labels = evaluate(MLP_MODEL, valid_loader)
    train_loss, train_probs, train_labels = evaluate(MLP_MODEL, train_loader)

    THRESHOLD = 0.5
    valid_preds = (valid_probs >= THRESHOLD).astype(int)

    cm = confusion_matrix(valid_labels, valid_preds)

    print(f"Train loss {train_loss:.4f} | Valid loss {valid_loss:.4f}")
    print(f"Valid ROC AUC: {roc_auc_score(valid_labels, valid_probs):.3f}")
    print(classification_report(valid_labels, valid_preds, digits=3))
    print(cm)

    # Validation Confusion Matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm, cmap="Greens")
    fig.colorbar(image, ax=ax)

    ax.set(
        xticks=np.arange(2),
        yticks=np.arange(2),
        xticklabels=["0", "1"],
        yticklabels=["0", "1"],
        xlabel="Predicted label",
        ylabel="True label",
        title="Validation Confusion Matrix",
    )

    for r in range(cm.shape[0]):
        for c in range(cm.shape[1]):
            ax.text(c, r, f"{cm[r, c]:,}", ha="center", va="center")

    plt.tight_layout()
    plt.savefig(f'images/optimize/MLP_Optimize-Validation_Confusion_Matrix-_{i}_-batchSize-{batch_size_}_epoch-{epoch_}_alpha-{alpha_}_weightDecay{weightDecay_}.png', dpi=300, bbox_inches='tight')
    plt.close("all")

    # Define and log current run parameters and results
    
    report = classification_report(valid_labels, valid_preds, output_dict=True)
    F1 = f1_score(valid_labels, valid_preds, pos_label=1)

    # Save model if F1 is better than current best
    if F1 > best_F1:
        best_F1 = F1
        torch.save(MLP_MODEL.state_dict(), "mlp_best.pt")

    runResults = {
        "Train Loss": train_loss,
        "Valid Loss": valid_loss,
        "Valid ROC AUC": roc_auc_score(valid_labels, valid_probs),
        "Classification Report": report,
        "Confusion Matrix": cm.tolist()
    }
    run_parameters = {
        "Index": i,
        "Batch Size": batch_size_,
        "Epochs": epoch_,
        "Alpha": alpha_,
        "Weight Decay": weightDecay_,
        "F1": F1,
        "Run Results": runResults
    }
    runLog.append(run_parameters)
    with open("optimize_runLog.json", "w", encoding="utf-8") as f:
        json.dump(runLog, f, indent=4)

    model_elapsed = perf_counter() - model_start # End timer
    print(f"Iteration elapsed: {model_elapsed:.2f}s")

    del (
        MLP_MODEL, MLP_MODEL_epoch_log, MLP_MODEL_loss_log,
        valid_loss, valid_probs, valid_labels, train_loss,
        train_probs, train_labels, valid_preds, cm,
        fig, ax, image,
        F1, runResults, run_parameters, model_elapsed
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    gc.collect()

 ==========     ==========     ==========     ==========
ITERATION [1/96]: 08:05:54 PM
    Batch Size:   512
    Epochs:       50
    α:            0.01
    Weight Decay: 0.01
    
Epoch 1/50, Loss: 1.5839 | 1.024s
Epoch 25/50, Loss: 0.3085 | 0.487s
Epoch 50/50, Loss: 0.1408 | 0.466s
Train loss 0.0766 | Valid loss 0.6427
Valid ROC AUC: 0.911
              precision    recall  f1-score   support

           0      0.985     0.793     0.879     13892
           1      0.312     0.886     0.462      1475

    accuracy                          0.802     15367
   macro avg      0.649     0.840     0.670     15367
weighted avg      0.920     0.802     0.839     15367

[[11015  2877]
 [  168  1307]]
Iteration elapsed: 25.81s
 ==========     ==========     ==========     ==========
ITERATION [2/96]: 08:06:20 PM
    Batch Size:   512
    Epochs:       50
    α:            0.01
    Weight Decay: 0.001
    
Epoch 1/50, Loss: 1.8649 | 0.405s
Epoch 25/50, Loss: 0.0491 | 0.457s
Epoch 50/50, Loss: 0.

In [20]:
f1_list = [item["F1"] for item in runLog]
max_index = max(range(len(f1_list)), key=f1_list.__getitem__)
max_index, f1_list[max_index]
f"[{max_index}] -> F1 Score: {f1_list[max_index]}"

'[27] -> F1 Score: 0.6203473945409429'

In [21]:
final_parameters = {
    "Index": runLog[max_index]['Index'],
    "Batch Size": runLog[max_index]['Batch Size'],
    "Epochs": runLog[max_index]['Epochs'],
    "Alpha": runLog[max_index]['Alpha'],
    "Weight Decay": runLog[max_index]['Weight Decay'],
    "F1": runLog[max_index]['F1'],
    "Run Results": runLog[max_index]['Run Results']
}
"""Saved parameters and results of run"""

final_parameters

{'Index': 27,
 'Batch Size': 1024,
 'Epochs': 150,
 'Alpha': 0.001,
 'Weight Decay': 0.001,
 'F1': 0.6203473945409429,
 'Run Results': {'Train Loss': 0.008153801495054633,
  'Valid Loss': 0.53246120429933,
  'Valid ROC AUC': 0.9484653281732688,
  'Classification Report': {'0': {'precision': 0.9919413329035378,
    'recall': 0.886049524906421,
    'f1-score': 0.9360100376411543,
    'support': 13892.0},
   '1': {'precision': 0.46484110885733604,
    'recall': 0.9322033898305084,
    'f1-score': 0.6203473945409429,
    'support': 1475.0},
   'accuracy': 0.8904795991410165,
   'macro avg': {'precision': 0.728391220880437,
    'recall': 0.9091264573684648,
    'f1-score': 0.7781787160910486,
    'support': 15367.0},
   'weighted avg': {'precision': 0.9413476691781426,
    'recall': 0.8904795991410165,
    'f1-score': 0.9057111895528605,
    'support': 15367.0}},
  'Confusion Matrix': [[12309, 1583], [100, 1375]]}}

### Dump results to new file

In [22]:
Path("contingency_parameters.txt").write_text("", encoding="utf-8")
with open("contingency_parameters.txt", "w", encoding="utf-8") as f:
    json.dump(final_parameters, f, indent=4)


## Finalized
Aims to isolate and rerun the model with the best F1.
With the best model saved, the model is loaded into `MLP_MODEL_FINAL`.
### Loading

In [23]:
MLP_MODEL_FINAL = MLP().to(device)
MLP_MODEL_FINAL.load_state_dict(torch.load("mlp_best.pt", map_location=device))

<All keys matched successfully>

### Evaluate Model

In [24]:
valid_loss, valid_probs, valid_labels = evaluate(MLP_MODEL_FINAL, valid_loader)
train_loss, train_probs, train_labels = evaluate(MLP_MODEL_FINAL, train_loader)

THRESHOLD = 0.5
valid_preds = (valid_probs >= THRESHOLD).astype(int)

cm = confusion_matrix(valid_labels, valid_preds)

print(f"Train loss {train_loss:.4f} | Valid loss {valid_loss:.4f}")
print(f"Valid ROC AUC: {roc_auc_score(valid_labels, valid_probs):.3f}")
print(classification_report(valid_labels, valid_preds, digits=3))
print(cm)

Train loss 0.0082 | Valid loss 0.5325
Valid ROC AUC: 0.948
              precision    recall  f1-score   support

           0      0.992     0.886     0.936     13892
           1      0.465     0.932     0.620      1475

    accuracy                          0.890     15367
   macro avg      0.728     0.909     0.778     15367
weighted avg      0.941     0.890     0.906     15367

[[12309  1583]
 [  100  1375]]


In [26]:
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

‼️‼️‼️**DO NOT** DO THE FOLLOWING‼️‼️‼️
- **DO NOT** `Shift` + `F1` (personal shortcut for restarting/rerunning)
- **DO NOT** RESTART the server
- **DO NOT** RESET the server

✔️✔️✔️OK to do✔️✔️✔️
- `Ctrl` + `Alt` + `Enter` cells below

## Test